# Whisper RTP Vietnamese ASR + Translation Pipeline
**Cải thiện:** Silero VAD, Context-Aware Translation, Jitter Buffer nâng cao, initial_prompt cho Whisper

Pipeline: `RTP → Jitter Buffer → Silero VAD → PhoWhisper-large → Qwen2.5-1.5B Translate`

## 0. Capture & Stream Setup (Windows - chạy ngoài Colab)
```bash
# cd "C:\Program Files\Wireshark"
# .\dumpcap.exe -i 11 -w "C:\Users\IMS-DPT\Desktop\number_counting.pcap"
# cd "C:\Users\IMS-DPT\Downloads\ffmpeg-8.1.1-essentials_build\ffmpeg-8.1.1-essentials_build\bin"
# .\ffmpeg.exe -re -i  -vn -acodec pcm_alaw -ar 8000 -ac 1 -f rtp rtp://127.0.0.1:10000
```

In [1]:
!apt-get update && apt-get install -y ffmpeg tcpreplay
!pip install faster-whisper scapy nest-asyncio pydub soundfile pynvml transformers accelerate jiwer sacrebleu silero-vad bert-score

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:7 https://cli.github.com/packages stable/main amd64 Packages [354 B]       
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,695 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 ht

## 1. Load Models: PhoWhisper + Qwen2.5-1.5B + Silero VAD

In [ ]:
# (Whisper + Silero VAD)
import socket
import threading
import subprocess
import time
import asyncio
import audioop
import heapq
import numpy as np
import nest_asyncio
import torch
from faster_whisper import WhisperModel
from pynvml import *
from transformers import AutoModelForCausalLM, AutoTokenizer

nest_asyncio.apply()

UDP_IP = "127.0.0.1"
UDP_PORT = 10000
SAMPLE_RATE = 16000
JITTER_BUFFER_SIZE = 5

# 1. Faster-PhoWhisper (~3.3GB VRAM)
print("Loading Whisper-large (CT2 format)...")
model_asr = WhisperModel("kiendt/PhoWhisper-large-ct2", device="cuda", compute_type="int8_float16")

# 2. Systran Whisper-large-v3 for Multilingual Translation (~3.5GB VRAM)
print("Loading Systran Whisper-large-v3 for Translation...")
model_trans = WhisperModel("Systran/faster-whisper-large-v3", device="cuda", compute_type="int8_float16")

# 3. Silero VAD (thay thế Energy-based VAD)
print("Loading Silero VAD...")
from silero_vad import load_silero_vad
silero_vad_model = load_silero_vad()
print("All models loaded!")

nvmlInit()
nvml_handle = nvmlDeviceGetHandleByIndex(0)

/tmp/ipykernel_58/3885613420.py:7: DeprecationWarning: 'audioop' is deprecated and slated for removal in Python 3.13
  import audioop


Loading Whisper-large (CT2 format)...


tokenizer.json: 0.00B [00:00, ?B/s]

Loading Systran Whisper-large-v3 for Translation...
Loading Silero VAD...
All models loaded!


## 2. Translation với Context Window

In [3]:
# # Context window để giữ ngữ cảnh giữa các câu liên tiếp
# translation_context = []

# def translate_vi_to_en(text, context_window=2):
#     """Dịch VI→EN có context từ các câu trước."""
#     if not text.strip():
#         return ""

#     # Lấy context từ các câu trước (sliding window)
#     context_str = " ".join(translation_context[-context_window:]) if translation_context else ""
#     context_block = f"Previous context:\n{context_str}\n\n" if context_str else ""

#     messages = [
#         {
#             "role": "system",
#             "content": (
#                 "You are a professional Vietnamese-to-English translator specializing in spoken language translation. "
#                 "Translate the transcript into natural, fluent, and conversational English. "
#                 "Preserve the original meaning, tone, emotion, and speaking style. "
#                 "Use the previous context (if any) to maintain coherence and consistency across sentences. "
#                 "Clean up minor transcription errors when necessary, but do not invent information. "
#                 "Do not explain the translation. Output only the English translation."
#             )
#         },
#         {
#             "role": "user",
#             "content": f"{context_block}Current transcript:\n{text}"
#         }
#     ]

#     text_prompt = llm_tokenizer.apply_chat_template(
#         messages,
#         tokenize=False,
#         add_generation_prompt=True
#     )

#     model_inputs = llm_tokenizer([text_prompt], return_tensors="pt").to("cuda")

#     max_new_tokens = max(100, int(len(text.split()) * 2.5))  # vi→en expands ~1.5–2x

#     with torch.no_grad():
#         generated_ids = llm_model.generate(
#             **model_inputs,
#             num_beams=3,
#             max_new_tokens=max_new_tokens,
#             repetition_penalty=1.15,
#             do_sample=False
#         )

#     generated_ids = [
#         output_ids[len(input_ids):]
#         for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
#     ]

#     translated_text = llm_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

#     # Cập nhật context window
#     translation_context.append(text)

#     return translated_text

## 3. Evaluation (Ground Truth)

In [4]:
import re
import jiwer
import sacrebleu
from bert_score import score

# =========================================================
# Ground Truth References
# =========================================================
GROUND_TRUTH_VI = (
    "Alo... cho mình gặp bộ phận kỹ thuật với ạ. "
    "Nhà mình đang gặp sự cố với đường truyền internet. "
    "Mạng chập chờn từ tối hôm qua tới giờ, lúc được lúc mất, mà chủ yếu là không kết nối được luôn. "
    "Mình đã thử tắt modem rồi bật lại vài lần, rút dây, đợi năm phút rồi cắm lại, nhưng tình hình vẫn không cải thiện. "
    "Thi thoảng thì có mạng một chút, nhưng rất yếu, gần như không thể mở nổi trang web nào cả. "
    "Mình cũng thử kiểm tra bằng điện thoại, bằng cả Wi-Fi lẫn bốn gờ để chắc chắn là không phải lỗi thiết bị, và chỉ có mạng Wi-Fi là bị. "
    "Địa chỉ nhà mình là số bốn mươi lăm, đường Lê Văn Sỹ, phường mười bốn, quận ba, Thành phố Hồ Chí Minh. "
    "Mình đăng ký dịch vụ internet gói SuperNet hai, nếu mình nhớ không nhầm. "
    "Tên chủ hợp đồng là Trần Minh Khoa, số điện thoại là không chín không tám một hai ba bốn năm sáu. "
    "Mình gọi lên chỉ mong bên kỹ thuật có thể kiểm tra giúp và báo lại sớm, chứ hôm nay mình cần làm việc trực tuyến cả ngày, mà mạng kiểu này thì bó tay rồi. "
    "À, còn một chuyện nữa, gần nhà mình có một công trình đang thi công. "
    "Không biết có phải do ảnh hưởng dây cáp hay không, nhưng mình nghi là liên quan vì trước giờ rất ổn định. "
    "Nếu cần hỗ trợ thêm thông tin hoặc có kỹ thuật viên đến kiểm tra thì cứ gọi số mình nha, mình luôn sẵn sàng. "
    "Cảm ơn nhiều ạ!"
)

# Ref 1: Direct translation (Your original)
GROUND_TRUTH_EN_1 = (
    "Hello... could I speak to the technical department, please? "
    "My house is experiencing an issue with the internet connection. "
    "The network has been unstable since last night, fluctuating on and off, but mostly there is no connection at all. "
    "I tried turning the modem off and on a few times, unplugging the cable, waiting for five minutes, and plugging it back in, but the situation still hasn't improved. "
    "Occasionally the internet comes back for a bit, but it's very weak, making it almost impossible to open any website. "
    "I also tried checking with my phone, using both Wi-Fi and 4G to ensure it wasn't a device error, and only the Wi-Fi network is affected. "
    "My address is number forty-five, Le Van Sy Street, Ward fourteen, District three, Ho Chi Minh City. "
    "I registered for the SuperNet two internet package, if I remember correctly. "
    "The contract holder's name is Tran Minh Khoa, and the phone number is zero nine zero eight one two three four five six. "
    "I am calling just hoping the technical team can check it out and report back soon, because I need to work online all day today, and with this kind of network, I'm completely helpless. "
    "Ah, there's one more thing: there is a construction site operating near my house. "
    "I don't know if it has affected the cables, but I suspect it's related because it has been very stable until now. "
    "If you need more information or if a technician comes to inspect it, just call my number, I am always available. "
    "Thank you very much!"
)

# Ref 2: Natural / Conversational Support Call
GROUND_TRUTH_EN_2 = (
    "Hi... can I get through to technical support, please? "
    "I'm having trouble with my home internet connection. "
    "The connection has been spotty since last night, going in and out, but mainly it's just completely disconnected. "
    "I've tried turning the modem off and back on a few times, unplugging it, waiting five minutes and plugging it back in, but nothing has changed. "
    "Sometimes it comes back slightly, but it's incredibly weak, so much so that I can barely load a single webpage. "
    "I even checked on my phone using both Wi-Fi and 4G just to be sure it wasn't my device, and it's definitely only the Wi-Fi that's acting up. "
    "My address is number forty-five, Le Van Sy Street, Ward fourteen, District three, Ho Chi Minh City. "
    "I believe I'm on the SuperNet two internet plan, if memory serves. "
    "The account holder's name is Tran Minh Khoa, and the phone number is zero nine zero eight one two three four five six. "
    "I'm calling in the hope that your technicians can look into it and get back to me soon, as I have to work online all day today, and I'm totally stuck with this connection. "
    "Oh, one more thing, there's a construction site working near my house. "
    "I'm not sure if it damaged the cables, but I suspect there's a connection since my internet was perfectly stable before this. "
    "If you need any more details or if someone's coming over to check, just give me a call, I am always ready. "
    "Thanks a lot!"
)

# Ref 3: More Formal / Direct Translation
GROUND_TRUTH_EN_3 = (
    "Hello... I would like to speak with the technical department. "
    "We are having an issue with the internet connection at my house. "
    "The network has been unstable since last night, intermittently dropping, but primarily failing to connect altogether. "
    "I have attempted restarting the modem several times, unplugging the power cord, waiting five minutes before plugging it back in, yet the situation remains unimproved. "
    "Occasionally there is a slight signal, but it is extremely weak and nearly impossible to browse any websites. "
    "I also tested it on my mobile phone, using both Wi-Fi and 4G to verify it is not a hardware fault, and indeed only the Wi-Fi is experiencing issues. "
    "The address is number forty-five, Le Van Sy Street, Ward fourteen, District three, Ho Chi Minh City. "
    "I am subscribed to the SuperNet two internet package, if I recall correctly. "
    "The name on the contract is Tran Minh Khoa, and the contact number is zero nine zero eight one two three four five six. "
    "I am calling to request that the technical team investigate this and notify me as soon as possible, because I have to work online for the entire day today, and this network condition makes it impossible. "
    "Also, one more detail, there is a construction project underway near my home. "
    "I am unsure if it has impacted the cables, but I strongly suspect it is related since the connection used to be very stable. "
    "Should you require further information or if a technician needs to visit for an inspection, please call my number, I am readily available. "
    "Thank you so much!"
)

# Combine references into a list
MULTIPLE_GROUND_TRUTH_EN = [GROUND_TRUTH_EN_1, GROUND_TRUTH_EN_2, GROUND_TRUTH_EN_3]

# =========================================================
# Evaluation Function
# =========================================================
def evaluate_system_performance(collected_vi_list, collected_en_list):
    if not collected_vi_list or not collected_en_list:
        print("Không có dữ liệu để đánh giá.")
        return

    hypothesis_vi = " ".join(collected_vi_list).strip()
    hypothesis_en = " ".join(collected_en_list).strip()

    print("\n================ SYSTEM BENCHMARK REPORT ================")
    print(f"Original VI:  {GROUND_TRUTH_VI[:80]}...")
    print(f"Predict VI:   {hypothesis_vi[:80]}...\n")

    # 1. Đánh giá ASR (Speech-to-Text) - Giữ nguyên
    def clean_and_normalize(text):
        text = text.lower()
        text = re.sub(r'[^\w\s]', '', text)
        return " ".join(text.split())

    clean_truth_vi = clean_and_normalize(GROUND_TRUTH_VI)
    clean_hyp_vi   = clean_and_normalize(hypothesis_vi)

    wer_score = jiwer.wer(clean_truth_vi, clean_hyp_vi)
    cer_score = jiwer.cer(clean_truth_vi, clean_hyp_vi)

    print(f"📊 [ASR Metrics]         WER: {wer_score * 100:.2f}% | CER: {cer_score * 100:.2f}%")

    # 2. Đánh giá Dịch thuật bằng ngữ nghĩa (BERTScore)
    print("\nĐang tính toán Semantic Score (BERTScore)...")
    
    # Định dạng dữ liệu cho BERTScore (So sánh 1 Hypothesis với 3 References)
    cands = [hypothesis_en]
    refs = [MULTIPLE_GROUND_TRUTH_EN] 

    # Chạy BERTScore (lang="en" vì đích là tiếng Anh)
    P, R, F1 = score(cands, refs, lang="en", verbose=False)

    print(f"[Semantic Metric]     BERTScore (F1):    {F1.mean().item():.4f}")
    print(f"[Semantic Metric]     BERTScore (Prec):  {P.mean().item():.4f}")
    print(f"[Semantic Metric]     BERTScore (Recall):{R.mean().item():.4f}")
    
    # (Tùy chọn) Giữ lại BLEU để xem chênh lệch giữa đếm chữ và đếm ý
    formatted_references = [[ref] for ref in MULTIPLE_GROUND_TRUTH_EN]
    bleu = sacrebleu.corpus_bleu(cands, formatted_references)
    print(f"[Lexical Metric]      BLEU Score:        {bleu.score:.2f}")
    
    print("=========================================================\n")

## 4. Receiver: Silero VAD + Jitter Buffer nâng cao

In [5]:
# Receiver với Silero VAD + cải thiện Jitter Buffer
def decode_pcma_chunk(payload_bytes):
    """Decode G.711 A-law (8kHz) to PCM Float32 (16kHz)"""
    if not payload_bytes:
        return np.array([], dtype=np.float32)

    pcm_8k  = audioop.alaw2lin(payload_bytes, 2)
    pcm_16k, _ = audioop.ratecv(pcm_8k, 2, 1, 8000, 16000, None)
    return np.frombuffer(pcm_16k, dtype=np.int16).astype(np.float32) / 32768.0


def silero_has_speech(audio_chunk: np.ndarray, threshold: float = 0.5) -> bool:
    """Kiểm tra xem chunk có tiếng nói thật sự không dùng Silero VAD."""
    from silero_vad import get_speech_timestamps
    audio_tensor = torch.from_numpy(audio_chunk)
    timestamps = get_speech_timestamps(
        audio_tensor,
        silero_vad_model,
        sampling_rate=SAMPLE_RATE,
        threshold=threshold
    )
    return len(timestamps) > 0


# async def jitter_buffer_and_stt(sock):
#     """Receive packets, Silero VAD, split sentences, STT và translate."""
#     print("[Receiver] Socket is listening for RTP (Silero VAD)...")

#     rtp_heap = []
#     current_pcm_samples = []

#     # --- VAD Settings (Energy sơ bộ để skip hoàn toàn silence) ---
#     SILENCE_THRESHOLD        = 0.012          # RMS sơ bộ
#     SILENCE_TIMEOUT_SAMPLES  = int(0.6 * SAMPLE_RATE)
#     MAX_AUDIO_SAMPLES        = int(15.0 * SAMPLE_RATE)
#     MIN_AUDIO_SAMPLES        = int(2.5 * SAMPLE_RATE)

#     # --- Jitter buffer: bỏ packet lỗi thời ---
#     MAX_SEQ_GAP   = 50   # sequence number gap tối đa trước khi discard
#     last_seq_pop  = None

#     consecutive_silence_samples = 0
#     has_speech = False

#     sock.settimeout(5.0)

#     # Domain hint cho Whisper (tuỳ dataset)
#     WHISPER_DOMAIN_HINTS = {
#         "banking": "Đây là cuộc gọi dịch vụ khách hàng ngân hàng. Số tài khoản, tên chủ tài khoản.",
#         "internet": "Đây là cuộc gọi hỗ trợ kỹ thuật internet. Modem, Wi-Fi, đường truyền.",
#         "friend":   "Đây là đoạn hội thoại thông thường giữa bạn bè.",
#     }
#     # Đổi key tuỳ theo PCAP_PATH đang dùng
#     active_domain = "internet"
#     initial_prompt = WHISPER_DOMAIN_HINTS.get(active_domain, "")

#     try:
#         while True:
#             try:
#                 data, addr = sock.recvfrom(2048)

#                 if len(data) < 12:
#                     continue

#                 seq_num = int.from_bytes(data[2:4], byteorder='big')
#                 payload = data[12:]

#                 heapq.heappush(rtp_heap, (seq_num, payload))

#                 if len(rtp_heap) >= JITTER_BUFFER_SIZE:
#                     popped_seq, active_payload = heapq.heappop(rtp_heap)

#                     # Bỏ packet lỗi thời (stale)
#                     if last_seq_pop is not None and (popped_seq - last_seq_pop) > MAX_SEQ_GAP:
#                         print(f"[Jitter] Discarding stale packet seq={popped_seq} (gap={popped_seq - last_seq_pop})")
#                         last_seq_pop = popped_seq
#                         continue
#                     last_seq_pop = popped_seq

#                     decoded_pcm = decode_pcma_chunk(active_payload)

#                     if len(decoded_pcm) > 0:
#                         rms = np.sqrt(np.mean(decoded_pcm ** 2))
#                         current_pcm_samples.extend(decoded_pcm)

#                         if rms < SILENCE_THRESHOLD:
#                             consecutive_silence_samples += len(decoded_pcm)
#                         else:
#                             consecutive_silence_samples = 0
#                             has_speech = True

#                     trigger_split = False

#                     if has_speech and consecutive_silence_samples >= SILENCE_TIMEOUT_SAMPLES:
#                         trigger_split = True
#                     elif len(current_pcm_samples) >= MAX_AUDIO_SAMPLES:
#                         trigger_split = True

#                     if trigger_split:
#                         if len(current_pcm_samples) < MIN_AUDIO_SAMPLES:
#                             consecutive_silence_samples = 0
#                             continue

#                         audio_chunk = np.array(current_pcm_samples, dtype=np.float32)
#                         current_pcm_samples = []
#                         consecutive_silence_samples = 0
#                         has_speech = True if (len(audio_chunk) / SAMPLE_RATE >= 9.5) else False

#                         # ---- Silero VAD: xác nhận có tiếng nói trước khi STT ----
#                         if not silero_has_speech(audio_chunk):
#                             print("[Silero VAD] Chunk không có tiếng nói – bỏ qua.")
#                             continue

#                         inf_start = time.time()

#                         segments, info = model.transcribe(
#                             audio_chunk,
#                             task="transcribe",
#                             language="vi",
#                             vad_filter=False,      # Silero VAD đã lọc ngoài
#                             beam_size=5,
#                             temperature=0.0,
#                             initial_prompt=initial_prompt   # Domain hint
#                         )

#                         text_vi = "".join([seg.text for seg in segments]).strip()

#                         if text_vi:
#                             text_en  = translate_vi_to_en(text_vi)
#                             latency  = (time.time() - inf_start) * 1000
#                             duration = len(audio_chunk) / SAMPLE_RATE

#                             global_collected_vi.append(text_vi)
#                             global_collected_en.append(text_en)

#                             print(f"[VAD STT & Trans] Length: {duration:.1f}s | Latency: {latency:.0f}ms")
#                             print(f"   ∟ [VI]: {text_vi}")
#                             print(f"   ∟ [EN]: {text_en}\n")

#             except socket.timeout:
#                 if len(current_pcm_samples) > 0 and has_speech:
#                     audio_chunk = np.array(current_pcm_samples, dtype=np.float32)

#                     # Silero VAD check cho chunk cuối
#                     if silero_has_speech(audio_chunk):
#                         segments, _ = model.transcribe(
#                             audio_chunk,
#                             language="vi",
#                             task="transcribe",
#                             vad_filter=False,
#                             beam_size=5,
#                             temperature=0.0,
#                             initial_prompt=initial_prompt
#                         )
#                         text_vi = "".join([seg.text for seg in segments]).strip()

#                         if text_vi:
#                             text_en = translate_vi_to_en(text_vi)
#                             global_collected_vi.append(text_vi)
#                             global_collected_en.append(text_en)
#                             print(f"[VAD Final Chunk] {len(audio_chunk) / SAMPLE_RATE:.1f}s")
#                             print(f"   ∟ [VI]: {text_vi}")
#                             print(f"   ∟ [EN]: {text_en}\n")

#                 print("[Receiver] No new packets received – stopping.")
#                 break
#     finally:
#         sock.close()

In [6]:
async def jitter_buffer_and_stt(sock):
    """Receive packets, Silero VAD, split sentences, STT và translate."""
    print("[Receiver] Socket is listening for RTP (Silero VAD)...")

    rtp_heap = []
    current_pcm_samples = []

    # --- VAD Settings (Energy sơ bộ để skip hoàn toàn silence) ---
    SILENCE_THRESHOLD        = 0.012          # RMS sơ bộ
    SILENCE_TIMEOUT_SAMPLES  = int(0.6 * SAMPLE_RATE)
    MAX_AUDIO_SAMPLES        = int(15.0 * SAMPLE_RATE)
    MIN_AUDIO_SAMPLES        = int(2.5 * SAMPLE_RATE)

    # --- Jitter buffer: bỏ packet lỗi thời ---
    MAX_SEQ_GAP   = 100   # Increased to 100 to reduce unnecessary packet drops
    last_seq_pop  = None

    consecutive_silence_samples = 0
    has_speech = False

    sock.settimeout(5.0)

    # Domain hint cho Whisper (tuỳ dataset)
    WHISPER_DOMAIN_HINTS = {
        "banking": "Đây là cuộc gọi dịch vụ khách hàng ngân hàng. Số tài khoản, tên chủ tài khoản.",
        "internet": "Đây là cuộc gọi hỗ trợ kỹ thuật internet. Modem, Wi-Fi, đường truyền.",
        "friend":   "Đây là đoạn hội thoại thông thường giữa bạn bè.",
    }
    active_domain = "internet"
    initial_prompt = WHISPER_DOMAIN_HINTS.get(active_domain, "")

    try:
        while True:
            try:
                data, addr = sock.recvfrom(2048)

                if len(data) < 12:
                    continue

                seq_num = int.from_bytes(data[2:4], byteorder='big')
                payload = data[12:]

                heapq.heappush(rtp_heap, (seq_num, payload))

                if len(rtp_heap) >= JITTER_BUFFER_SIZE:
                    popped_seq, active_payload = heapq.heappop(rtp_heap)

                    # Bỏ packet lỗi thời (stale)
                    if last_seq_pop is not None and (popped_seq - last_seq_pop) > MAX_SEQ_GAP:
                        print(f"[Jitter] Discarding stale packet seq={popped_seq} (gap={popped_seq - last_seq_pop})")
                        last_seq_pop = popped_seq
                        continue
                    last_seq_pop = popped_seq

                    decoded_pcm = decode_pcma_chunk(active_payload)

                    if len(decoded_pcm) > 0:
                        rms = np.sqrt(np.mean(decoded_pcm ** 2))
                        current_pcm_samples.extend(decoded_pcm)

                        if rms < SILENCE_THRESHOLD:
                            consecutive_silence_samples += len(decoded_pcm)
                        else:
                            consecutive_silence_samples = 0
                            has_speech = True

                    trigger_split = False

                    if has_speech and consecutive_silence_samples >= SILENCE_TIMEOUT_SAMPLES:
                        trigger_split = True
                    elif len(current_pcm_samples) >= MAX_AUDIO_SAMPLES:
                        trigger_split = True

                    if trigger_split:
                        if len(current_pcm_samples) < MIN_AUDIO_SAMPLES:
                            consecutive_silence_samples = 0
                            continue

                        audio_chunk = np.array(current_pcm_samples, dtype=np.float32)
                        current_pcm_samples = []
                        consecutive_silence_samples = 0
                        has_speech = True if (len(audio_chunk) / SAMPLE_RATE >= 9.5) else False

                        # ---- Silero VAD: xác nhận có tiếng nói trước khi STT ----
                        if not silero_has_speech(audio_chunk):
                            print("[Silero VAD] Chunk không có tiếng nói – bỏ qua.")
                            continue

                        inf_start = time.time()

                        # --- PASS 1: PhoWhisper ASR ---
                        segments_vi, _ = model_asr.transcribe(
                            audio_chunk,
                            task="transcribe",
                            language="vi",
                            vad_filter=False, 
                            beam_size=5,
                            temperature=0.0,
                            initial_prompt=initial_prompt
                        )
                        text_vi = "".join([seg.text for seg in segments_vi]).strip()

                        if text_vi:
                            # --- PASS 2: Systran Translation ---
                            segments_en, _ = model_trans.transcribe(
                                audio_chunk,
                                task="translate", 
                                vad_filter=False,
                                beam_size=5,
                                temperature=0.0,
                                repetition_penalty=1.2,
                                no_repeat_ngram_size=3,
                                compression_ratio_threshold=2.0,
                                log_prob_threshold=-1.0
                            )
                            text_en = "".join([seg.text for seg in segments_en]).strip()
                            
                            latency  = (time.time() - inf_start) * 1000
                            duration = len(audio_chunk) / SAMPLE_RATE

                            global_collected_vi.append(text_vi)
                            global_collected_en.append(text_en)

                            print(f"[Whisper Dual-Pass] Length: {duration:.1f}s | Latency: {latency:.0f}ms")
                            print(f"   ∟ [VI]: {text_vi}")
                            print(f"   ∟ [EN]: {text_en}\n")

            except socket.timeout:
                if len(current_pcm_samples) > 0 and has_speech:
                    audio_chunk = np.array(current_pcm_samples, dtype=np.float32)

                    # Silero VAD check cho chunk cuối
                    if silero_has_speech(audio_chunk):
                        inf_start = time.time()  # Fixed missing timestamp

                        # --- PASS 1 (Final Chunk): PhoWhisper ---
                        segments_vi, _ = model_asr.transcribe(
                            audio_chunk,
                            language="vi",
                            task="transcribe",
                            vad_filter=False,
                            beam_size=5,
                            temperature=0.0,
                            initial_prompt=initial_prompt
                        )
                        text_vi = "".join([seg.text for seg in segments_vi]).strip()

                        if text_vi:
                            # --- PASS 2 (Final Chunk): Systran ---
                            segments_en, _ = model_trans.transcribe(
                                audio_chunk,
                                task="translate", 
                                vad_filter=False,
                                beam_size=5,
                                temperature=0.0,
                                repetition_penalty=1.2,
                                no_repeat_ngram_size=3,
                                compression_ratio_threshold=2.0,
                                log_prob_threshold=-1.0
                            )
                            text_en = "".join([seg.text for seg in segments_en]).strip()
                            
                            latency  = (time.time() - inf_start) * 1000
                            duration = len(audio_chunk) / SAMPLE_RATE
                            
                            global_collected_vi.append(text_vi)
                            global_collected_en.append(text_en)
                            print(f"[Whisper Dual-Pass Final] Length: {duration:.1f}s | Latency: {latency:.0f}ms")
                            print(f"   ∟ [VI]: {text_vi}")
                            print(f"   ∟ [EN]: {text_en}\n")

                print("[Receiver] No new packets received – stopping.")
                break
    finally:
        sock.close()

## 5. Sender: Python Network Replay từ PCAP

In [7]:
import socket
import threading
import asyncio
import time
from scapy.all import rdpcap, UDP

# Chọn file PCAP tương ứng với domain
# PCAP_PATH = "/kaggle/input/datasets/fzl208/banking-trouble/banking_trouble.pcap"
# PCAP_PATH = "/kaggle/input/datasets/fzl208/friend/friend.pcap"
PCAP_PATH  = "/kaggle/input/datasets/fzl208/internet-trouble/internet_trouble.pcap"
TARGET_IP  = "127.0.0.1"
TARGET_PORT = 10000


def python_network_replay(pcap_path, target_ip, target_port):
    print(f"[Sender] Reading file: {pcap_path}...")
    packets    = rdpcap(pcap_path)
    sender_sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

    print(f"[Sender] Starting RTP stream to {target_ip}:{target_port}...")
    last_pkt_time = None

    for pkt in packets:
        if pkt.haslayer(UDP):
            current_pkt_time = float(pkt.time)

            if last_pkt_time is not None:
                sleep_time = current_pkt_time - last_pkt_time
                if sleep_time > 0:
                    time.sleep(sleep_time)

            last_pkt_time = current_pkt_time
            raw_payload   = bytes(pkt[UDP].payload)

            if len(raw_payload) > 0:
                sender_sock.sendto(raw_payload, (target_ip, target_port))

    print("[Sender] Finished sending all packets!")
    sender_sock.close()

## 6. Main

In [8]:
def main():
    global global_collected_vi, global_collected_en
    global_collected_vi = []
    global_collected_en = []

    # Reset context window dịch thuật cho mỗi lần chạy
    # translation_context.clear()

    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

    try:
        sock.bind((TARGET_IP, TARGET_PORT))
        print(f"[Receiver] Successfully bound socket at {TARGET_IP}:{TARGET_PORT}")
    except Exception as e:
        print(f"Cannot bind socket: {e}")
        return

    sender_thread = threading.Thread(
        target=python_network_replay,
        args=(PCAP_PATH, TARGET_IP, TARGET_PORT)
    )
    sender_thread.start()

    print("\n--- Starting Live VAD Simulation (Silero VAD) ---")
    asyncio.run(jitter_buffer_and_stt(sock))
    print("--- End Live VAD Simulation ---\n")

    evaluate_system_performance(global_collected_vi, global_collected_en)


if __name__ == "__main__":
    main()

[Receiver] Successfully bound socket at 127.0.0.1:10000
[Sender] Reading file: /kaggle/input/datasets/fzl208/internet-trouble/internet_trouble.pcap...

--- Starting Live VAD Simulation (Silero VAD) ---
[Receiver] Socket is listening for RTP (Silero VAD)...
[Sender] Starting RTP stream to 127.0.0.1:10000...
[Jitter] Discarding stale packet seq=3309 (gap=3303)
[Jitter] Discarding stale packet seq=3391 (gap=3385)
[Jitter] Discarding stale packet seq=3470 (gap=3464)
[Whisper Dual-Pass] Length: 14.9s | Latency: 4751ms
   ∟ [VI]: lô cho mình vào bộ phận kỹ thuật với ạ nhà mình đang gặp sự cố lưu truyền tin internet mạng chập chờn từ tối hôm qua đến giờ lô lượng lúc mất và chủ yếu là không kết nối được luôn mình đã thử tắt modem bật lại vài lần rút dây đợi nửa phút rồi cắm lại nhưng tình hình vẫn không cải thiện.
   ∟ [EN]: Hello, welcome to the technical part of this video. My family is having an accident with internet connection It has been a long time since last night The problem is that w

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[Semantic Metric]     BERTScore (F1):    0.9009
[Semantic Metric]     BERTScore (Prec):  0.9052
[Semantic Metric]     BERTScore (Recall):0.8967
[Lexical Metric]      BLEU Score:        25.32

